# 08 — GRU Sequence Model (Phase 3)

Hourly sepsis prediction with a **GRU** over each patient's ICU time series.

- Same patient split as the baselines (`results/train|val|test_patient_ids.csv`)
- Same 40 input features as XGBoost / Random Forest, plus missingness indicators
- Same target: `SepsisLabel` **for every hour** (one prediction per row)
- Same metrics at threshold 0.5, so results are directly comparable

The GRU is **unidirectional**, and missing values are only forward-filled, so the prediction at hour *t* uses only data from hours ≤ *t* (no peeking into the future).

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # fall back to CPU for any op MPS doesn't support

import sys
import time
import random
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# Project root
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import TRAIN_A, TRAIN_B, RESULTS_DIR, MODELS_DIR

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

# Plot style shared by every chart below
sns.set_theme(style="whitegrid", context="notebook")
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984"

print("Training Set A:", TRAIN_A)
print("Training Set B:", TRAIN_B)
print("Results:", RESULTS_DIR)
print("Models:", MODELS_DIR)
print("Device:", DEVICE)

## 1. Patient split (same as the baselines)

In [ ]:
train_ids = pd.read_csv(RESULTS_DIR / "train_patient_ids.csv")["patient_id"].tolist()
val_ids = pd.read_csv(RESULTS_DIR / "val_patient_ids.csv")["patient_id"].tolist()
test_ids = pd.read_csv(RESULTS_DIR / "test_patient_ids.csv")["patient_id"].tolist()

print("Training patients:", len(train_ids))
print("Validation patients:", len(val_ids))
print("Test patients:", len(test_ids))

all_patient_files = {}
for file_path in TRAIN_A.glob("*.psv"):
    all_patient_files[file_path.stem] = file_path
for file_path in TRAIN_B.glob("*.psv"):
    all_patient_files[file_path.stem] = file_path

print("\nTotal patient files found:", len(all_patient_files))
for name, ids in [("training", train_ids), ("validation", val_ids), ("test", test_ids)]:
    missing = [pid for pid in ids if pid not in all_patient_files]
    print(f"Missing {name} patients:", len(missing))

## 2. Load each patient as a sequence

Raw arrays are cached in `dataset/processed/` (git-ignored), so re-running the notebook skips the slow file loading.

In [ ]:
TARGET = "SepsisLabel"

# Read the column order from one file
_sample = pd.read_csv(next(iter(all_patient_files.values())), sep="|")
FEATURE_COLUMNS = [c for c in _sample.columns if c != TARGET]

# Demographic / admin columns: no missingness indicator for these
STATIC_COLUMNS = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]
MEASURED_COLUMNS = [c for c in FEATURE_COLUMNS if c not in STATIC_COLUMNS]
MEASURED_IDX = [FEATURE_COLUMNS.index(c) for c in MEASURED_COLUMNS]

print("Raw features:", len(FEATURE_COLUMNS))
print("Measured (vitals + labs):", len(MEASURED_COLUMNS))

CACHE_DIR = PROJECT_ROOT / "dataset" / "processed"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def load_split(patient_ids, split_name):
    # Returns {patient_id: (X float32 [hours, 40] with NaNs, y int8 [hours])}
    cache_path = CACHE_DIR / f"gru_raw_{split_name}.pkl"
    if cache_path.exists():
        with open(cache_path, "rb") as f:
            data = pickle.load(f)
        print(f"{split_name}: loaded {len(data):,} patients from cache")
        return data

    data = {}
    for i, pid in enumerate(patient_ids):
        df = pd.read_csv(all_patient_files[pid], sep="|")
        X = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).to_numpy(np.float32)
        y = df[TARGET].to_numpy(np.int8)
        data[pid] = (X, y)
        if (i + 1) % 2000 == 0:
            print(f"{split_name}: loaded {i + 1:,} / {len(patient_ids):,} patients")

    with open(cache_path, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"{split_name}: done, cached to {cache_path.name}")
    return data


raw_train = load_split(train_ids, "train")
raw_val = load_split(val_ids, "val")
raw_test = load_split(test_ids, "test")

for name, raw in [("Train", raw_train), ("Val", raw_val), ("Test", raw_test)]:
    lengths = np.array([len(y) for _, y in raw.values()])
    rows = lengths.sum()
    pos = sum(int(y.sum()) for _, y in raw.values())
    print(f"{name}: {len(raw):,} patients | {rows:,} hours | positive hours {pos:,} ({pos / rows * 100:.2f}%) "
          f"| length median {int(np.median(lengths))}, max {lengths.max()}")

## 3. Preprocessing (statistics from training patients only)

1. **Missingness indicators** for the 34 vitals/labs (1 = measured this hour). Whether a lab was ordered is itself informative.
2. **Forward-fill** within each patient (carry the last known value forward; never backward, to avoid leaking future values).
3. Fill anything still missing with the **training median**.
4. **Standardise** with training mean/std and clip to ±10.

In [ ]:
def ffill_patient(X):
    # np.array(..., copy) makes a writable copy (pandas may return a read-only view)
    return np.array(pd.DataFrame(X).ffill().to_numpy(), dtype=np.float32, copy=True)


# Statistics from training patients only
train_filled = np.concatenate([ffill_patient(X) for X, _ in raw_train.values()], axis=0)
with np.errstate(all="ignore"):
    medians = np.nanmedian(train_filled, axis=0).astype(np.float32)
medians = np.nan_to_num(medians, nan=0.0)

nan_mask = np.isnan(train_filled)
train_filled[nan_mask] = np.take(medians, np.where(nan_mask)[1])
means = train_filled.mean(axis=0).astype(np.float32)
stds = train_filled.std(axis=0).astype(np.float32)
stds[stds < 1e-6] = 1.0
del train_filled, nan_mask

CLIP = 10.0


def preprocess(raw):
    out = {}
    for pid, (X, y) in raw.items():
        observed = (~np.isnan(X[:, MEASURED_IDX])).astype(np.float32)
        Xf = ffill_patient(X)
        nan_mask = np.isnan(Xf)
        Xf[nan_mask] = np.take(medians, np.where(nan_mask)[1])
        Xf = np.clip((Xf - means) / stds, -CLIP, CLIP)
        out[pid] = (np.concatenate([Xf, observed], axis=1).astype(np.float32), y.astype(np.float32))
    return out


train_data = preprocess(raw_train)
val_data = preprocess(raw_val)
test_data = preprocess(raw_test)

INPUT_SIZE = next(iter(train_data.values()))[0].shape[1]
print("Input features per hour:", INPUT_SIZE, f"({len(FEATURE_COLUMNS)} values + {len(MEASURED_COLUMNS)} missingness flags)")

## 4. Dataset and batching

Patients have very different stay lengths, so batches are built from patients of similar length (less padding = faster). Padded hours are masked out of the loss and the metrics.

In [ ]:
class PatientSequenceDataset(Dataset):
    def __init__(self, data, patient_ids):
        self.patient_ids = list(patient_ids)
        self.X = [torch.from_numpy(data[pid][0]) for pid in self.patient_ids]
        self.y = [torch.from_numpy(data[pid][1]) for pid in self.patient_ids]
        self.lengths = np.array([len(y) for y in self.y])

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def collate_batch(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs])
    X = pad_sequence(xs, batch_first=True)
    Y = pad_sequence(ys, batch_first=True)
    mask = torch.arange(X.size(1))[None, :] < lengths[:, None]
    return X, Y, mask


class BucketBatchSampler(Sampler):
    # Groups patients of similar length into the same batch
    def __init__(self, lengths, batch_size, shuffle=True, bucket_multiplier=50, seed=SEED):
        self.lengths = np.asarray(lengths)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.chunk = batch_size * bucket_multiplier
        self.rng = np.random.default_rng(seed)

    def __iter__(self):
        idx = np.arange(len(self.lengths))
        if self.shuffle:
            self.rng.shuffle(idx)
        batches = []
        for start in range(0, len(idx), self.chunk):
            chunk = idx[start:start + self.chunk]
            chunk = chunk[np.argsort(self.lengths[chunk], kind="stable")]
            batches += [chunk[i:i + self.batch_size].tolist() for i in range(0, len(chunk), self.batch_size)]
        if self.shuffle:
            self.rng.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return int(np.ceil(len(self.lengths) / self.batch_size))


BATCH_SIZE = 64

train_ds = PatientSequenceDataset(train_data, train_ids)
val_ds = PatientSequenceDataset(val_data, val_ids)
test_ds = PatientSequenceDataset(test_data, test_ids)

train_loader = DataLoader(train_ds, batch_sampler=BucketBatchSampler(train_ds.lengths, BATCH_SIZE, shuffle=True),
                          collate_fn=collate_batch)
val_loader = DataLoader(val_ds, batch_sampler=BucketBatchSampler(val_ds.lengths, 256, shuffle=False),
                        collate_fn=collate_batch)
test_loader = DataLoader(test_ds, batch_sampler=BucketBatchSampler(test_ds.lengths, 256, shuffle=False),
                         collate_fn=collate_batch)

print("Train batches per epoch:", len(train_loader))

## 5. GRU model

In [ ]:
class SepsisGRU(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        out, _ = self.gru(x)               # [batch, hours, hidden]
        return self.head(out).squeeze(-1)  # [batch, hours] logits, one per hour


MODEL_CONFIG = dict(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, dropout=0.2)
model = SepsisGRU(**MODEL_CONFIG).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTrainable parameters: {n_params:,}")

## 6. Training

Class imbalance is handled the same way as XGBoost: positive hours are weighted by `negative / positive` (≈54.5). Early stopping on validation ROC-AUC.

In [ ]:
neg = sum(int((y == 0).sum()) for _, y in train_data.values())
pos = sum(int((y == 1).sum()) for _, y in train_data.values())
pos_weight = neg / pos
print("Negative hours:", f"{neg:,}")
print("Positive hours:", f"{pos:,}")
print("pos_weight:", round(pos_weight, 2))

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE), reduction="none")


def masked_loss(logits, y, mask):
    loss = criterion(logits, y)
    mask = mask.float()
    return (loss * mask).sum() / mask.sum()


@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for X, Y, M in loader:
        logits = model(X.to(DEVICE)).float().cpu()
        probs = torch.sigmoid(logits)
        all_probs.append(probs[M].numpy())
        all_labels.append(Y[M].numpy())
    return np.concatenate(all_labels).astype(int), np.concatenate(all_probs)


EPOCHS = 30
PATIENCE = 4
LR = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

best_auc, best_epoch, epochs_without_improvement = -1.0, 0, 0
best_model_path = MODELS_DIR / "gru_baseline.pt"
history = []

print("Starting GRU training...\n")
for epoch in range(1, EPOCHS + 1):
    start = time.time()
    model.train()
    running_loss, n_batches = 0.0, 0

    for X, Y, M in train_loader:
        X, Y, M = X.to(DEVICE), Y.to(DEVICE), M.to(DEVICE)
        optimizer.zero_grad()
        loss = masked_loss(model(X), Y, M)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
        n_batches += 1

    train_loss = running_loss / n_batches
    y_val_true, y_val_prob = predict(model, val_loader)
    val_auc = roc_auc_score(y_val_true, y_val_prob)
    scheduler.step(val_auc)

    history.append({"epoch": epoch, "train_loss": train_loss, "val_auc": val_auc,
                    "lr": optimizer.param_groups[0]["lr"]})
    print(f"Epoch {epoch:2d} | train loss {train_loss:.4f} | val AUC {val_auc:.4f} | "
          f"lr {optimizer.param_groups[0]['lr']:.1e} | {time.time() - start:.0f}s")

    if val_auc > best_auc:
        best_auc, best_epoch, epochs_without_improvement = val_auc, epoch, 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "model_config": MODEL_CONFIG,
            "feature_columns": FEATURE_COLUMNS,
            "measured_columns": MEASURED_COLUMNS,
            "medians": medians,
            "means": means,
            "stds": stds,
            "clip": CLIP,
            "best_epoch": best_epoch,
            "best_val_auc": best_auc,
        }, best_model_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping: no improvement for {PATIENCE} epochs.")
            break

print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best validation AUC:", round(best_auc, 4))
print("Model saved to:", best_model_path)

history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_DIR / "gru_training_history.csv", index=False)

In [ ]:
# ------------------------------------------------------------
# Training curves: loss, validation ROC-AUC, learning rate
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", color=BLUE, linewidth=2)
axes[0].set_title("Training Loss (weighted BCE)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(history_df["epoch"], history_df["val_auc"], marker="o", color=ORANGE, linewidth=2)
axes[1].axvline(best_epoch, linestyle="--", color=GREY, label=f"Best epoch {best_epoch} (AUC = {best_auc:.4f})")
axes[1].set_title("Validation ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("ROC-AUC")
axes[1].legend(loc="lower right")

axes[2].step(history_df["epoch"], history_df["lr"], where="post", color=BLUE, linewidth=2)
axes[2].set_yscale("log")
axes[2].set_title("Learning Rate (ReduceLROnPlateau)")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Learning rate")

for ax in axes:
    ax.set_xticks(history_df["epoch"])

plt.tight_layout()
plt.show()

## 7. Validation results (best checkpoint, threshold 0.5)

In [ ]:
checkpoint = torch.load(best_model_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

THRESHOLD = 0.5


def evaluate(y_true, y_prob, title):
    y_pred = (y_prob >= THRESHOLD).astype(int)
    metrics = {
        "auc": roc_auc_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    cm = confusion_matrix(y_true, y_pred)
    print(f"===== {title} RESULTS =====")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"F1 Score : {metrics['f1']:.4f}")
    print(f"ROC-AUC  : {metrics['auc']:.4f}")
    print("\n===== CONFUSION MATRIX =====")
    print(cm)
    print("\n===== CLASSIFICATION REPORT =====")
    print(classification_report(y_true, y_pred, target_names=["No Sepsis", "Sepsis"], zero_division=0))
    return metrics, cm


y_val_true, y_val_prob = predict(model, val_loader)
print("Validation rows:", f"{len(y_val_true):,}", "(XGBoost used 231,472)\n")
val_metrics, val_cm = evaluate(y_val_true, y_val_prob, "VALIDATION")

pd.DataFrame([{
    "model": "GRU_baseline",
    "best_epoch": best_epoch,
    "validation_auc": val_metrics["auc"],
    "validation_accuracy": val_metrics["accuracy"],
    "validation_precision": val_metrics["precision"],
    "validation_recall": val_metrics["recall"],
    "validation_f1": val_metrics["f1"],
    "threshold": THRESHOLD,
}]).to_csv(RESULTS_DIR / "gru_validation_metrics.csv", index=False)
print("Saved:", RESULTS_DIR / "gru_validation_metrics.csv")

## 8. Test results

In [ ]:
y_test_true, y_test_prob = predict(model, test_loader)
print("Test rows:", f"{len(y_test_true):,}", "(XGBoost used 234,302)\n")
test_metrics, test_cm = evaluate(y_test_true, y_test_prob, "TEST")

pd.DataFrame([{
    "model": "GRU_baseline",
    "best_epoch": best_epoch,
    "test_auc": test_metrics["auc"],
    "test_accuracy": test_metrics["accuracy"],
    "test_precision": test_metrics["precision"],
    "test_recall": test_metrics["recall"],
    "test_f1": test_metrics["f1"],
    "threshold": THRESHOLD,
    "test_rows": len(y_test_true),
    "test_positive_samples": int(y_test_true.sum()),
    "test_negative_samples": int((y_test_true == 0).sum()),
}]).to_csv(RESULTS_DIR / "gru_test_metrics.csv", index=False)

pd.DataFrame(
    test_cm,
    index=["Actual_No_Sepsis", "Actual_Sepsis"],
    columns=["Predicted_No_Sepsis", "Predicted_Sepsis"],
).to_csv(RESULTS_DIR / "gru_test_confusion_matrix.csv")

print("Saved:", RESULTS_DIR / "gru_test_metrics.csv")
print("Saved:", RESULTS_DIR / "gru_test_confusion_matrix.csv")

## 9. Results visualization

Same layout as the XGBoost / Random Forest dashboards, plus curves that only make sense for a probabilistic sequence model:
ROC and precision–recall curves, a threshold sweep on validation, the predicted-risk distribution, and the risk trajectory around sepsis onset.

### 9.1 Summary tables, metrics and confusion matrix

In [ ]:
# ============================================================
# GRU — RESULTS DASHBOARD
# ============================================================

from IPython.display import display
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

# ------------------------------------------------------------
# 1. Validation vs test metrics
# ------------------------------------------------------------

METRIC_KEYS = {"ROC-AUC": "auc", "Accuracy": "accuracy", "Precision": "precision",
               "Recall": "recall", "F1 Score": "f1"}

gru_results_df = pd.DataFrame({
    "Metric": list(METRIC_KEYS),
    "Validation": [val_metrics[k] for k in METRIC_KEYS.values()],
    "Test": [test_metrics[k] for k in METRIC_KEYS.values()],
})

print("============================================================")
print("                    GRU BASELINE RESULTS")
print("============================================================")

display(
    gru_results_df.style
    .format({"Validation": "{:.4f}", "Test": "{:.4f}"})
    .set_caption(f"Validation vs Test Performance (threshold {THRESHOLD})")
)

# ------------------------------------------------------------
# 2. Model configuration
# ------------------------------------------------------------

config_df = pd.DataFrame({
    "Configuration": [
        "Input features per hour",
        "Hidden size / GRU layers / dropout",
        "Trainable parameters",
        "Training patients",
        "Training rows (hours)",
        "Validation patients",
        "Validation rows (hours)",
        "Test patients",
        "Test rows (hours)",
        "Positive training hours",
        "pos_weight",
        "Batch size / learning rate",
        "Best epoch",
        "Classification threshold",
    ],
    "Value": [
        INPUT_SIZE,
        f"{MODEL_CONFIG['hidden_size']} / {MODEL_CONFIG['num_layers']} / {MODEL_CONFIG['dropout']}",
        f"{n_params:,}",
        f"{len(train_ids):,}",
        f"{int(train_ds.lengths.sum()):,}",
        f"{len(val_ids):,}",
        f"{len(y_val_true):,}",
        f"{len(test_ids):,}",
        f"{len(y_test_true):,}",
        f"{pos:,}",
        round(pos_weight, 2),
        f"{BATCH_SIZE} / {LR}",
        best_epoch,
        THRESHOLD,
    ],
})

print("\nMODEL CONFIGURATION")
display(config_df)

# ------------------------------------------------------------
# 3. Performance comparison graph
# ------------------------------------------------------------

plot_df = gru_results_df.melt(id_vars="Metric", var_name="Dataset", value_name="Score")

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=plot_df, x="Metric", y="Score", hue="Dataset", palette=[BLUE, ORANGE])
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=9, padding=2)

plt.ylim(0, 1.05)
plt.title("GRU Baseline Performance")
plt.ylabel("Score")
plt.xlabel("")
plt.legend(title="Dataset")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 4. Validation and test confusion matrix heatmaps (counts + % of actual class)
# ------------------------------------------------------------

CM_INDEX = ["Actual: No Sepsis", "Actual: Sepsis"]
CM_COLUMNS = ["Predicted: No Sepsis", "Predicted: Sepsis"]

gru_val_cm_df = pd.DataFrame(val_cm, index=CM_INDEX, columns=CM_COLUMNS)
gru_test_cm_df = pd.DataFrame(test_cm, index=CM_INDEX, columns=CM_COLUMNS)

print("\nTEST CONFUSION MATRIX")
display(gru_test_cm_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (name, cm) in zip(axes, [("Validation", val_cm), ("Test", test_cm)]):
    row_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    cm_labels = np.array([[f"{n:,}\n({p:.1f}%)" for n, p in zip(r_n, r_p)] for r_n, r_p in zip(cm, row_pct)])
    sns.heatmap(pd.DataFrame(row_pct, index=CM_INDEX, columns=CM_COLUMNS), annot=cm_labels, fmt="",
                cmap="Blues", vmin=0, vmax=100, cbar=False, annot_kws={"fontsize": 12}, ax=ax)
    ax.set_title(f"GRU {name} Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

fig.suptitle(f"Threshold {THRESHOLD} — shading = % of actual class", y=0.98, fontsize=11)
plt.tight_layout()
plt.show()

### 9.2 ROC, precision–recall and threshold curves

Only ~1.8% of hours are positive, so the precision–recall curve is more informative than ROC here; the dashed line is the positive rate (a random model's precision).
The dot on each curve is the operating point at threshold 0.5. The threshold sweep uses **validation only**; all metrics reported above still use 0.5.

In [ ]:
SPLITS = [
    ("Validation", y_val_true, y_val_prob, BLUE),
    ("Test", y_test_true, y_test_prob, ORANGE),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ------------------------------------------------------------
# 1. ROC curves
# ------------------------------------------------------------

for name, y_true, y_prob, color in SPLITS:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f"{name} (AUC = {roc_auc_score(y_true, y_prob):.4f})")
    y_pred = y_prob >= THRESHOLD
    op_fpr = (y_pred & (y_true == 0)).sum() / (y_true == 0).sum()
    op_tpr = (y_pred & (y_true == 1)).sum() / (y_true == 1).sum()
    axes[0].scatter(op_fpr, op_tpr, s=70, color=color, edgecolor="white", linewidth=2, zorder=3)

axes[0].plot([0, 1], [0, 1], linestyle="--", color=GREY, label="Random baseline")
axes[0].set_title("ROC Curve — GRU")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(loc="lower right")

# ------------------------------------------------------------
# 2. Precision-recall curves
# ------------------------------------------------------------

for name, y_true, y_prob, color in SPLITS:
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    axes[1].plot(recall, precision, color=color, linewidth=2, label=f"{name} (AP = {ap:.4f})")
    axes[1].axhline(y_true.mean(), linestyle="--", color=color, linewidth=1,
                    label=f"{name} positive rate ({y_true.mean():.2%})")
    y_pred = (y_prob >= THRESHOLD).astype(int)
    axes[1].scatter(recall_score(y_true, y_pred), precision_score(y_true, y_pred),
                    s=70, color=color, edgecolor="white", linewidth=2, zorder=3)

axes[1].set_title("Precision–Recall Curve — GRU")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_ylim(0, 1)
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3. Threshold sweep on validation
# ------------------------------------------------------------

thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
val_pos = y_val_true == 1
sweep = []
for t in thresholds:
    pred = y_val_prob >= t
    tp = (pred & val_pos).sum()
    p = tp / pred.sum() if pred.sum() else 0.0
    r = tp / val_pos.sum()
    sweep.append({"Threshold": t, "Precision": p, "Recall": r, "F1": 2 * p * r / (p + r) if p + r else 0.0})
threshold_df = pd.DataFrame(sweep)
best_row = threshold_df.loc[threshold_df["F1"].idxmax()]

plt.figure(figsize=(10, 6))
for metric, color in [("Precision", BLUE), ("Recall", ORANGE), ("F1", AQUA)]:
    plt.plot(threshold_df["Threshold"], threshold_df[metric], color=color, linewidth=2, label=metric)
plt.axvline(THRESHOLD, linestyle="--", color=GREY, label=f"Current threshold ({THRESHOLD})")
plt.axvline(best_row["Threshold"], linestyle=":", color="black",
            label=f"Best-F1 threshold ({best_row['Threshold']:.2f})")
plt.title("Precision, Recall and F1 vs Threshold — Validation")
plt.xlabel("Classification threshold")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

print("THRESHOLD SWEEP (VALIDATION)")
display(
    pd.DataFrame([threshold_df.loc[threshold_df["Threshold"] == THRESHOLD].iloc[0], best_row],
                 index=["Current threshold", "Best-F1 threshold"])
    .style.format("{:.4f}")
)

### 9.3 Predicted risk and early warning

PhysioNet sets `SepsisLabel = 1` from **6 hours before** clinical sepsis onset, so hour 0 below is the first labelled hour, not the onset itself.
The trajectory plot shows how the GRU's predicted risk for septic test patients rises as that point approaches, against the average risk of non-septic patients.
It uses only septic patients first labelled at least 24 hours into their stay, so the same patients appear at every hour; about a quarter of septic patients are labelled from their first ICU hour and have no history before the label.

In [ ]:
@torch.no_grad()
def predict_by_patient(model, ds, batch_size=256):
    # {patient_id: (labels [hours], probabilities [hours])}
    model.eval()
    out = {}
    for idx in BucketBatchSampler(ds.lengths, batch_size, shuffle=False):
        X, _, _ = collate_batch([ds[i] for i in idx])
        probs = torch.sigmoid(model(X.to(DEVICE)).float().cpu()).numpy()
        for row, i in enumerate(idx):
            out[ds.patient_ids[i]] = (ds.y[i].numpy().astype(int), probs[row, :ds.lengths[i]])
    return out


test_by_patient = predict_by_patient(model, test_ds)

# ------------------------------------------------------------
# 1. Predicted risk distribution by true class (test)
# ------------------------------------------------------------

risk_df = pd.DataFrame({
    "Predicted risk": y_test_prob,
    "Actual": np.where(y_test_true == 1, "Sepsis", "No Sepsis"),
})

plt.figure(figsize=(10, 5.5))
sns.histplot(data=risk_df, x="Predicted risk", hue="Actual", hue_order=["No Sepsis", "Sepsis"],
             palette=[BLUE, ORANGE], bins=50, stat="density", common_norm=False,
             element="step", fill=True, alpha=0.25, linewidth=2)
plt.axvline(THRESHOLD, linestyle="--", color=GREY, label=f"Threshold ({THRESHOLD})")
plt.title("Distribution of Predicted Risk by Actual Class — Test")
plt.xlabel("Predicted probability of sepsis (hourly)")
plt.ylabel("Density (each class normalised separately)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 2. Risk trajectory relative to the first positive label (test)
# ------------------------------------------------------------

# Fixed cohort: septic patients whose first positive label is >= HOURS_BEFORE hours into the stay,
# so every patient contributes to every hour (patients labelled at admission would distort the curve).
# PhysioNet records end 9 hours after the first positive label.
HOURS_BEFORE, HOURS_AFTER = 24, 9
traj_rows = []
for pid, (y, p) in test_by_patient.items():
    if y.max() == 0:
        continue
    onset = int(np.argmax(y == 1))
    if onset < HOURS_BEFORE:
        continue
    rel = np.arange(len(y)) - onset
    keep = (rel >= -HOURS_BEFORE) & (rel <= HOURS_AFTER)
    traj_rows.append(pd.DataFrame({"hour": rel[keep], "prob": p[keep]}))

n_traj_patients = len(traj_rows)
traj = pd.concat(traj_rows).groupby("hour")["prob"].agg(
    median="median", q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75), patients="size")
non_septic_mean = np.mean([p.mean() for y, p in test_by_patient.values() if y.max() == 0])

plt.figure(figsize=(11, 5.5))
plt.fill_between(traj.index, traj["q25"], traj["q75"], color=ORANGE, alpha=0.2, label=f"Septic patients (n = {n_traj_patients}) — IQR")
plt.plot(traj.index, traj["median"], color=ORANGE, linewidth=2, label="Septic patients — median risk")
plt.axhline(non_septic_mean, color=BLUE, linewidth=2, label=f"Non-septic patients — mean risk ({non_septic_mean:.3f})")
plt.axhline(THRESHOLD, linestyle="--", color=GREY, label=f"Threshold ({THRESHOLD})")
plt.axvline(0, linestyle=":", color="black", label="First SepsisLabel = 1 (onset − 6 h)")
plt.title("GRU Predicted Risk Around Sepsis Label Onset — Test")
plt.xlabel("Hours relative to first positive label")
plt.ylabel("Predicted probability of sepsis")
plt.xlim(-HOURS_BEFORE, HOURS_AFTER)
plt.xticks(range(-HOURS_BEFORE, HOURS_AFTER + 1, 3))
plt.ylim(0, 1)
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3, frameon=False)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3. Patient-level alerting (any hour >= threshold)
# ------------------------------------------------------------

septic = [(y, p) for y, p in test_by_patient.values() if y.max() == 1]
non_septic = [(y, p) for y, p in test_by_patient.values() if y.max() == 0]
alerted_before = [(p[: np.argmax(y == 1)] >= THRESHOLD).any() for y, p in septic if np.argmax(y == 1) > 0]

patient_level_df = pd.DataFrame({
    "Measure": [
        "Septic test patients",
        "Non-septic test patients",
        "Septic patients with ≥1 alert during stay",
        "Septic patients alerted before first positive label (label not at admission)",
        "Non-septic patients with ≥1 alert (false alarm)",
    ],
    "Value": [
        f"{len(septic):,}",
        f"{len(non_septic):,}",
        f"{np.mean([(p >= THRESHOLD).any() for _, p in septic]):.2%}",
        f"{np.mean(alerted_before):.2%}",
        f"{np.mean([(p >= THRESHOLD).any() for _, p in non_septic]):.2%}",
    ],
})

display(patient_level_df.style.set_caption(f"Patient-Level Alerting — Test (threshold {THRESHOLD})"))

## 10. Comparison with the baselines (test set)

In [ ]:
rows = []
for name, file in [("XGBoost", "xgboost_test_metrics.csv"),
                   ("Random Forest", "random_forest_test_metrics.csv"),
                   ("GRU", "gru_test_metrics.csv")]:
    path = RESULTS_DIR / file
    if path.exists():
        m = pd.read_csv(path).iloc[0]
        rows.append({
            "Model": name,
            "ROC-AUC": m.get("test_auc"),
            "Accuracy": m.get("test_accuracy"),
            "Precision": m.get("test_precision"),
            "Recall": m.get("test_recall"),
            "F1": m.get("test_f1"),
        })

comparison = pd.DataFrame(rows).set_index("Model")
display(comparison.style.format("{:.4f}").set_caption("Test performance (threshold 0.5)"))

comparison_plot = comparison.reset_index().melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(11, 6))
ax = sns.barplot(data=comparison_plot, x="Metric", y="Score", hue="Model",
                 palette=[BLUE, ORANGE, AQUA][: len(comparison)])
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8, padding=2)
plt.ylim(0, 1.05)
plt.title("XGBoost vs Random Forest vs GRU — Test Set")
plt.xlabel("")
plt.ylabel("Score")
plt.legend(title="Model", loc="upper right")
plt.tight_layout()
plt.show()